In [344]:
# Cell 1 - Import library

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output

print("Libraries loaded.")

Libraries loaded.


In [345]:
# Cell 2 - Konfigurasi utama

# ============================================================
# FILE PATH
# ============================================================
# CLEAN_FILE_PATH = Path("/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Adelia/CSV/6.Csv") ("/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Bungkuk/Kanaya/CSV/2.Csv")
CLEAN_FILE_PATH = Path("/media/spell/Spell-lab/Lidar/A.Raw_Testing/Uncontrolled Room/Jatuh/Kanaya/CSV/2.Csv")
# ============================================================
# FRAME CONFIG
# ============================================================
TIMESTAMP_UNIT = "ns"      # diasumsikan nanosecond
FRAME_DURATION_MS = 100    # 100 ms

# ============================================================
# SORT OPTION
# True  = sort berdasarkan Timestamp
# False = pakai urutan baris asli CSV
# ============================================================
USE_SORT_TIMESTAMP = True

# ============================================================
# ROI CONFIG (original coordinates)
# ============================================================
ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

# ============================================================
# VISUAL CONFIG
# ============================================================
POINT_SIZE = 2
MAX_POINTS_PER_FRAME = 30000

# Fixed axis agar visual tidak menipu
FIXED_X_RANGE = [0.0, 3.0]
FIXED_Y_RANGE = [-1.5, 1.5]
FIXED_Z_RANGE = [0.0, 2.0]

print("===== CONFIG =====")
print(f"CLEAN_FILE_PATH     : {CLEAN_FILE_PATH}")
print(f"FILE EXISTS         : {CLEAN_FILE_PATH.exists()}")
print(f"TIMESTAMP_UNIT      : {TIMESTAMP_UNIT}")
print(f"FRAME_DURATION_MS   : {FRAME_DURATION_MS}")
print(f"USE_SORT_TIMESTAMP  : {USE_SORT_TIMESTAMP}")
print(f"ROI X               : {ROI_X_MIN} to {ROI_X_MAX}")
print(f"ROI Y               : {ROI_Y_MIN} to {ROI_Y_MAX}")
print(f"ROI Z               : {ROI_Z_MIN} to {ROI_Z_MAX}")

===== CONFIG =====
CLEAN_FILE_PATH     : /media/spell/Spell-lab/Lidar/A.Raw_Testing/Uncontrolled Room/Jatuh/Kanaya/CSV/2.Csv
FILE EXISTS         : True
TIMESTAMP_UNIT      : ns
FRAME_DURATION_MS   : 100
USE_SORT_TIMESTAMP  : True
ROI X               : 0.0 to 3.0
ROI Y               : -1.5 to 1.5
ROI Z               : 0.0 to 2.0


In [346]:
# Cell 3 - Load dataset

REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity"]

if not CLEAN_FILE_PATH.exists():
    raise FileNotFoundError(f"File not found: {CLEAN_FILE_PATH}")

df_raw = pd.read_csv(CLEAN_FILE_PATH)

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Simpan urutan baris asli untuk audit non-sort
df_raw["_orig_row_id"] = np.arange(len(df_raw))

# Konversi numeric
for col in REQUIRED_COLUMNS:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

before_drop = len(df_raw)
df_raw = df_raw.dropna(subset=REQUIRED_COLUMNS).copy()
after_drop = len(df_raw)

print("===== DATA LOADED =====")
print(f"Rows before dropna : {before_drop:,}")
print(f"Rows after dropna  : {after_drop:,}")
print(f"Dropped rows       : {before_drop - after_drop:,}")
print(f"Unique timestamp   : {df_raw['Timestamp'].nunique():,}")

display(df_raw.head())

/tmp/ipykernel_99794/2209196399.py:8: DtypeWarning:

Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.



===== DATA LOADED =====
Rows before dropna : 1,150,753
Rows after dropna  : 1,150,752
Dropped rows       : 1
Unique timestamp   : 11,987


,Version,Handle,LiDAR Index,Rsvd,Error Code,Timestamp Type,Data Type,Timestamp,X,Y,...,Reflectivity,Tag,Ori_x,Ori_y,Ori_z,Ori_radius,Ori_theta,Ori_phi,FrameCounter,_orig_row_id
1,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.166,1.673,...,71.0,0.0,-1.166,1.673,2.028,0.0,0.0,0.0,0.0,1
2,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.145,1.555,...,75.0,0.0,-1.145,1.555,2.038,0.0,0.0,0.0,0.0,2
3,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.114,1.441,...,84.0,0.0,-1.114,1.441,2.057,0.0,0.0,0.0,0.0,3
4,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.088,1.331,...,88.0,0.0,-1.088,1.331,2.075,0.0,0.0,0.0,0.0,4
5,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.196,1.635,...,71.0,0.0,-1.196,1.635,2.011,0.0,0.0,0.0,0.0,5


In [347]:
# Cell 4 - Apply sort / non-sort

if USE_SORT_TIMESTAMP:
    df_work = df_raw.sort_values(["Timestamp", "_orig_row_id"]).reset_index(drop=True).copy()
    ORDER_MODE = "SORTED_BY_TIMESTAMP"
else:
    df_work = df_raw.sort_values("_orig_row_id").reset_index(drop=True).copy()
    ORDER_MODE = "ORIGINAL_ROW_ORDER"

print("===== ORDER MODE =====")
print(f"ORDER_MODE : {ORDER_MODE}")

display(df_work.head())

===== ORDER MODE =====
ORDER_MODE : SORTED_BY_TIMESTAMP


,Version,Handle,LiDAR Index,Rsvd,Error Code,Timestamp Type,Data Type,Timestamp,X,Y,...,Reflectivity,Tag,Ori_x,Ori_y,Ori_z,Ori_radius,Ori_theta,Ori_phi,FrameCounter,_orig_row_id
0,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.166,1.673,...,71.0,0.0,-1.166,1.673,2.028,0.0,0.0,0.0,0.0,1
1,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.145,1.555,...,75.0,0.0,-1.145,1.555,2.038,0.0,0.0,0.0,0.0,2
2,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.114,1.441,...,84.0,0.0,-1.114,1.441,2.057,0.0,0.0,0.0,0.0,3
3,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.088,1.331,...,88.0,0.0,-1.088,1.331,2.075,0.0,0.0,0.0,0.0,4
4,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549882092940,-1.196,1.635,...,71.0,0.0,-1.196,1.635,2.011,0.0,0.0,0.0,0.0,5


In [348]:
# Cell 5 - Buat temp_frame_id berbasis absolute time

if TIMESTAMP_UNIT == "ns":
    frame_duration_ts = int(FRAME_DURATION_MS * 1_000_000)   # 100 ms = 100,000,000 ns
elif TIMESTAMP_UNIT == "us":
    frame_duration_ts = int(FRAME_DURATION_MS * 1_000)       # 100 ms = 100,000 us
elif TIMESTAMP_UNIT == "ms":
    frame_duration_ts = int(FRAME_DURATION_MS)               # 100 ms = 100 ms
elif TIMESTAMP_UNIT == "s":
    frame_duration_ts = FRAME_DURATION_MS / 1000.0           # 0.1 s
else:
    raise ValueError("TIMESTAMP_UNIT harus salah satu dari: ns, us, ms, s")

t0 = df_work["Timestamp"].min()

if frame_duration_ts <= 0:
    raise ValueError("frame_duration_ts invalid.")

df_work["temp_frame_id"] = ((df_work["Timestamp"] - t0) // frame_duration_ts).astype(int)

print("===== TEMP FRAME INFO =====")
print(f"Timestamp start     : {t0}")
print(f"Frame duration (ts) : {frame_duration_ts}")
print(f"Total temp frames   : {df_work['temp_frame_id'].nunique():,}")

display(
    df_work[["Timestamp", "temp_frame_id", "X", "Y", "Z", "Reflectivity"]].head(10)
)

===== TEMP FRAME INFO =====
Timestamp start     : 6549882092940
Frame duration (ts) : 100000000
Total temp frames   : 58


,Timestamp,temp_frame_id,X,Y,Z,Reflectivity
0,6549882092940,0,-1.166,1.673,2.028,71.0
1,6549882092940,0,-1.145,1.555,2.038,75.0
2,6549882092940,0,-1.114,1.441,2.057,84.0
3,6549882092940,0,-1.088,1.331,2.075,88.0
4,6549882092940,0,-1.196,1.635,2.011,71.0
5,6549882092940,0,-1.170,1.517,2.020,66.0
6,6549882092940,0,-1.132,1.400,2.035,90.0
7,6549882092940,0,-1.105,1.296,2.061,73.0
8,6549882092940,0,-1.226,1.596,1.993,73.0
9,6549882092940,0,-1.198,1.484,2.011,71.0


In [349]:
# Cell 6 - ROI filtering saja

roi_mask = (
    (df_work["X"] >= ROI_X_MIN) & (df_work["X"] <= ROI_X_MAX) &
    (df_work["Y"] >= ROI_Y_MIN) & (df_work["Y"] <= ROI_Y_MAX) &
    (df_work["Z"] >= ROI_Z_MIN) & (df_work["Z"] <= ROI_Z_MAX)
)

df_roi = df_work.loc[roi_mask].copy()

print("===== ROI FILTER RESULT =====")
print(f"Total rows before ROI : {len(df_work):,}")
print(f"Total rows after ROI  : {len(df_roi):,}")
print(f"Retained ratio        : {len(df_roi) / len(df_work):.6f}")

display(df_roi.head())

===== ROI FILTER RESULT =====
Total rows before ROI : 1,150,752
Total rows after ROI  : 105,450
Retained ratio        : 0.091636


,Version,Handle,LiDAR Index,Rsvd,Error Code,Timestamp Type,Data Type,Timestamp,X,Y,...,Tag,Ori_x,Ori_y,Ori_z,Ori_radius,Ori_theta,Ori_phi,FrameCounter,_orig_row_id,temp_frame_id
662,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549884972940,0.000,0.000,...,0.0,0.000,-0.000,0.000,0.0,0.0,0.0,0.0,663,0
663,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549884972940,0.000,0.000,...,0.0,0.000,-0.000,0.000,0.0,0.0,0.0,0.0,664,0
665,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549884972940,0.000,0.000,...,0.0,0.000,-0.000,0.000,0.0,0.0,0.0,0.0,666,0
666,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549884972940,0.576,-0.236,...,0.0,0.576,-0.236,0.581,0.0,0.0,0.0,0.0,667,0
667,0,50440384.0,1.0,0.0,0.0,0.0,2.0,6549884972940,0.598,-0.236,...,0.0,0.598,-0.236,0.554,0.0,0.0,0.0,0.0,668,0


In [350]:
# Cell 7 - Ringkasan per frame ROI

frame_summary = (
    df_roi.groupby("temp_frame_id")
    .agg(
        n_points=("temp_frame_id", "size"),
        timestamp_min=("Timestamp", "min"),
        timestamp_max=("Timestamp", "max"),
        x_med=("X", "median"),
        y_med=("Y", "median"),
        z_med=("Z", "median"),
        x_mean=("X", "mean"),
        y_mean=("Y", "mean"),
        z_mean=("Z", "mean"),
    )
    .reset_index()
    .sort_values("temp_frame_id")
    .reset_index(drop=True)
)

# Hitung centroid jump
frame_summary["dx"] = frame_summary["x_med"].diff()
frame_summary["dy"] = frame_summary["y_med"].diff()
frame_summary["dz"] = frame_summary["z_med"].diff()

frame_summary["dxyz"] = np.sqrt(
    frame_summary["dx"]**2 +
    frame_summary["dy"]**2 +
    frame_summary["dz"]**2
)

frame_ids = frame_summary["temp_frame_id"].tolist()

print("===== FRAME SUMMARY =====")
print(f"Total ROI frames available : {len(frame_ids):,}")

display(frame_summary.head(10))

===== FRAME SUMMARY =====
Total ROI frames available : 58


,temp_frame_id,n_points,timestamp_min,timestamp_max,x_med,y_med,z_med,x_mean,y_mean,z_mean,dx,dy,dz,dxyz
0,0,2394,6549884972940,6549979532940,0.6580,0.0,0.2020,0.904820,0.008750,0.337511,NaN,NaN,NaN,NaN
1,1,2563,6549984332940,6550078912940,0.6760,0.0,0.1950,0.926363,-0.018051,0.328397,0.0180,0.0,-0.0070,0.019313
2,2,2485,6550084192940,6550178372940,0.6720,0.0,0.2050,0.900756,-0.009417,0.324292,-0.0040,0.0,0.0100,0.010770
3,3,2564,6550183172940,6550277792940,0.6650,0.0,0.2070,0.918206,-0.005300,0.329967,-0.0070,0.0,0.0020,0.007280
4,4,2544,6550282112940,6550381992940,0.6575,0.0,0.1970,0.887434,0.013164,0.319871,-0.0075,0.0,-0.0100,0.012500
5,5,2634,6550382472940,6550481852940,0.6435,0.0,0.2115,0.903373,0.021883,0.337546,-0.0140,0.0,0.0145,0.020156
6,6,2444,6550487132940,6550581232940,0.6635,0.0,0.2025,0.946244,0.028570,0.333625,0.0200,0.0,-0.0090,0.021932
7,7,2481,6550586512940,6550681132940,0.6520,0.0,0.1930,0.933139,0.036954,0.334678,-0.0115,0.0,-0.0095,0.014916
8,8,2447,6550685932940,6550780512940,0.6560,0.0,0.2020,0.931556,0.035486,0.332494,0.0040,0.0,0.0090,0.009849
9,9,2417,6550785312940,6550879892940,0.6570,0.0,0.2020,0.934988,0.035945,0.322680,0.0010,0.0,0.0000,0.001000


In [351]:
# Cell 8 - Fungsi visualisasi 3D ROI-only

def plot_roi_frame(temp_frame_id):
    frame_df = df_roi[df_roi["temp_frame_id"] == temp_frame_id].copy()

    if len(frame_df) == 0:
        fig = go.Figure()
        fig.update_layout(
            title=f"ROI Only | temp_frame_id={temp_frame_id} | EMPTY FRAME",
            scene=dict(
                xaxis=dict(title="X", range=FIXED_X_RANGE),
                yaxis=dict(title="Y", range=FIXED_Y_RANGE),
                zaxis=dict(title="Z", range=FIXED_Z_RANGE),
                aspectmode="manual",
                aspectratio=dict(x=1.5, y=1.5, z=1.0),
            ),
            width=900,
            height=700,
        )
        return fig

    if len(frame_df) > MAX_POINTS_PER_FRAME:
        frame_df = frame_df.sample(MAX_POINTS_PER_FRAME, random_state=42)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=frame_df["X"],
            y=frame_df["Y"],
            z=frame_df["Z"],
            mode="markers",
            marker=dict(
                size=POINT_SIZE,
                color=frame_df["Z"],
                colorscale="Turbo",
                opacity=0.80,
                colorbar=dict(title="Z"),
            ),
            text=[
                f"temp_frame_id={temp_frame_id}<br>"
                f"Timestamp={ts}<br>"
                f"X={x:.3f}<br>Y={y:.3f}<br>Z={z:.3f}<br>R={r}"
                for ts, x, y, z, r in zip(
                    frame_df["Timestamp"],
                    frame_df["X"],
                    frame_df["Y"],
                    frame_df["Z"],
                    frame_df["Reflectivity"],
                )
            ],
            hoverinfo="text",
            name="ROI points",
        )
    )

    fig.update_layout(
        title=(
            f"Clean Dataset ROI Only | {ORDER_MODE}<br>"
            f"temp_frame_id = {temp_frame_id}"
        ),
        scene=dict(
            xaxis=dict(title="X", range=FIXED_X_RANGE),
            yaxis=dict(title="Y", range=FIXED_Y_RANGE),
            zaxis=dict(title="Z", range=FIXED_Z_RANGE),
            aspectmode="manual",
            aspectratio=dict(x=1.5, y=1.5, z=1.0),
        ),
        width=900,
        height=700,
        margin=dict(l=0, r=0, b=0, t=80),
    )

    return fig

In [352]:
# Cell 9 - Fungsi summary frame

def get_frame_summary_row(temp_frame_id):
    row = frame_summary[frame_summary["temp_frame_id"] == temp_frame_id]
    if len(row) == 0:
        return pd.DataFrame({"metric": ["status"], "value": ["EMPTY FRAME"]})
    
    row = row.iloc[0]

    summary_df = pd.DataFrame({
        "metric": [
            "ORDER_MODE",
            "temp_frame_id",
            "n_points",
            "timestamp_min",
            "timestamp_max",
            "x_med",
            "y_med",
            "z_med",
            "x_mean",
            "y_mean",
            "z_mean",
            "dx_from_prev",
            "dy_from_prev",
            "dz_from_prev",
            "dxyz_from_prev",
        ],
        "value": [
            ORDER_MODE,
            int(row["temp_frame_id"]),
            int(row["n_points"]),
            row["timestamp_min"],
            row["timestamp_max"],
            float(row["x_med"]),
            float(row["y_med"]),
            float(row["z_med"]),
            float(row["x_mean"]),
            float(row["y_mean"]),
            float(row["z_mean"]),
            np.nan if pd.isna(row["dx"]) else float(row["dx"]),
            np.nan if pd.isna(row["dy"]) else float(row["dy"]),
            np.nan if pd.isna(row["dz"]) else float(row["dz"]),
            np.nan if pd.isna(row["dxyz"]) else float(row["dxyz"]),
        ]
    })

    return summary_df

In [353]:
# Cell 10 - GUI manual Prev / Next

if len(frame_ids) == 0:
    raise ValueError("Tidak ada frame ROI yang bisa divisualisasikan.")

current_index = 0

prev_button = widgets.Button(description="Prev Frame", icon="arrow-left")
next_button = widgets.Button(description="Next Frame", icon="arrow-right")

frame_info = widgets.HTML()
plot_output = widgets.Output()
summary_output = widgets.Output()

def render_frame():
    global current_index

    temp_frame_id = frame_ids[current_index]

    frame_info.value = (
        f"<b>File:</b> {CLEAN_FILE_PATH}<br>"
        f"<b>Order mode:</b> {ORDER_MODE}<br>"
        f"<b>Frame:</b> {current_index + 1} / {len(frame_ids)} "
        f"(<b>temp_frame_id:</b> {temp_frame_id})"
    )

    with plot_output:
        clear_output(wait=True)
        fig = plot_roi_frame(temp_frame_id)
        fig.show()

    with summary_output:
        clear_output(wait=True)
        display(get_frame_summary_row(temp_frame_id))

def on_prev_clicked(b):
    global current_index
    if current_index > 0:
        current_index -= 1
        render_frame()

def on_next_clicked(b):
    global current_index
    if current_index < len(frame_ids) - 1:
        current_index += 1
        render_frame()

prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

controls = widgets.HBox([prev_button, next_button])

display(controls)
display(frame_info)
display(summary_output)
display(plot_output)

render_frame()

HTML(value='')

Output()

Output()

In [354]:
# Cell 11 - Tampilkan frame dengan jump terbesar

print("===== FRAMES WITH LARGEST ROI CENTROID JUMPS =====")

display(
    frame_summary[
        [
            "temp_frame_id",
            "n_points",
            "timestamp_min",
            "timestamp_max",
            "x_med",
            "y_med",
            "z_med",
            "dx",
            "dy",
            "dz",
            "dxyz",
        ]
    ]
    .sort_values("dxyz", ascending=False)
    .head(20)
)

===== FRAMES WITH LARGEST ROI CENTROID JUMPS =====


,temp_frame_id,n_points,timestamp_min,timestamp_max,x_med,y_med,z_med,dx,dy,dz,dxyz
17,17,1693,6551582172940,6551681532940,2.2150,0.0,0.2110,1.4480,0.0,0.0900,1.450794
57,57,723,6555603632940,6555631952940,2.6720,0.0,1.1970,0.0590,0.0,0.7920,0.794195
34,34,1508,6553282832940,6553378352940,2.5660,0.0,0.4435,-0.1000,0.0,-0.7040,0.711067
33,33,1428,6553183472940,6553279952940,2.6660,0.0,1.1475,0.0490,0.0,0.6775,0.679270
31,31,1509,6552984272940,6553080752940,2.6360,0.0,0.5880,0.0410,0.0,0.1740,0.178765
18,18,1769,6551685852940,6551780892940,2.3720,0.0,0.2150,0.1570,0.0,0.0040,0.157051
16,16,1919,6551482332940,6551581692940,0.7670,0.0,0.1210,0.1510,0.0,0.0310,0.154149
24,24,1614,6552282492940,6552381852940,2.6235,0.0,0.4420,0.0745,0.0,0.1320,0.151573
19,19,1613,6551785212940,6551880252940,2.4820,0.0,0.2790,0.1100,0.0,0.0640,0.127264
32,32,1545,6553083632940,6553180592940,2.6170,0.0,0.4700,-0.0190,0.0,-0.1180,0.119520


In [355]:
# Cell 12 - Jump manual ke frame tertentu

def jump_to_temp_frame(target_temp_frame_id):
    global current_index

    if target_temp_frame_id not in frame_ids:
        print(f"temp_frame_id {target_temp_frame_id} tidak ditemukan.")
        return

    current_index = frame_ids.index(target_temp_frame_id)
    render_frame()
    print(f"Jumped to temp_frame_id={target_temp_frame_id}")

# Contoh:
# jump_to_temp_frame(8)

In [356]:
all_frame_ids = sorted(df_work["temp_frame_id"].unique())
roi_frame_ids = sorted(df_roi["temp_frame_id"].unique())

missing_roi_frames = sorted(set(all_frame_ids) - set(roi_frame_ids))

print("All temp frames:", len(all_frame_ids))
print("ROI temp frames:", len(roi_frame_ids))
print("Missing ROI frames:", len(missing_roi_frames))
print("First 30 missing ROI frames:", missing_roi_frames[:30])

print("All frame range:", min(all_frame_ids), "to", max(all_frame_ids))
print("ROI frame range:", min(roi_frame_ids), "to", max(roi_frame_ids))

All temp frames: 58
ROI temp frames: 58
Missing ROI frames: 0
First 30 missing ROI frames: []
All frame range: 0 to 57
ROI frame range: 0 to 57


In [357]:
temp_time_summary = (
    df_work.groupby("temp_frame_id")
    .agg(
        n_points=("temp_frame_id", "size"),
        timestamp_min=("Timestamp", "min"),
        timestamp_max=("Timestamp", "max"),
    )
    .reset_index()
    .sort_values("temp_frame_id")
)

temp_time_summary["dt_from_prev"] = temp_time_summary["timestamp_min"].diff()

display(
    temp_time_summary[
        ["temp_frame_id", "n_points", "timestamp_min", "timestamp_max", "dt_from_prev"]
    ].sort_values("dt_from_prev", ascending=False).head(20)
)

,temp_frame_id,n_points,timestamp_min,timestamp_max,dt_from_prev
5,5,19968,6550382472940,6550481852940,100360000.0
9,9,19968,6550782432940,6550881812940,100340000.0
1,1,19968,6549982412940,6550081792940,100320000.0
18,18,19968,6551682492940,6551781852940,100320000.0
54,54,19968,6555282512940,6555381872940,100320000.0
51,51,19968,6554982512940,6555081872940,100320000.0
48,48,19968,6554682512940,6554781872940,100320000.0
45,45,19968,6554382512940,6554481872940,100320000.0
42,42,19968,6554082512940,6554181872940,100320000.0
39,39,19968,6553782512940,6553881872940,100320000.0
